# Forecasting basics with honest backtesting

Data Scientist track, lesson DS12. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** split a daily series into trend, weekday pattern and noise, backtest a forecast the way it will be used, beat a seasonal naive baseline or say that it cannot be beaten, build lag and rolling features that never read the future, and compare every forecast with the dataset's own expected values.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
from sklearn.linear_model import LinearRegression

SEED = 2026       # the seed of the whole Data Scientist track

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")


con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people, 177,379 events, and 21,871 signup_complete events.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)
daily = con.sql("SELECT CAST(event_ts AS DATE) AS day, COUNT(*) AS signups FROM events WHERE event_name = 'signup_complete' GROUP BY 1 ORDER BY 1").df()
signups = daily.set_index(pd.to_datetime(daily["day"]))["signups"].asfreq("D").astype(float)      # one number for each day: the signups of that UTC day
assert int(signups.sum()) == 21871
print(f"{len(signups)} days, {signups.index[0].date()} to {signups.index[-1].date()}, {int(signups.sum()):,} signups, {signups.min():.0f} to {signups.max():.0f} a day.")


## The weeks and the backtest

The series is `signups`. `backtest` forecasts one week at a time from the days before it, and `weekly_mae` scores each week. Run this cell as it is.

In [ ]:
df = pd.DataFrame({"y": signups})
WEEKS = [m for m in df.index[df.index.dayofweek == 0] if m + pd.Timedelta(days=6) <= df.index[-1]]       # every Monday that starts a full week
FIRST = 8                    # the first 8 weeks are history only: the forecasts start with the 9th week, March 2


def backtest(forecast):
    """Rolling origin: for each week from the 9th, forecast its 7 days from the days before it only, and set the forecast beside what happened."""
    rows = []
    for k, monday in enumerate(WEEKS):
        if k < FIRST:
            continue
        days = pd.date_range(monday, periods=7)
        past = df.loc[: monday - pd.Timedelta(days=1)]                      # everything up to the Sunday night before the week
        rows.append(pd.DataFrame({"week": monday, "day": days, "actual": df.loc[days, "y"].to_numpy(), "forecast": np.asarray(forecast(past, days), dtype=float)}))
    return pd.concat(rows, ignore_index=True)


def weekly_mae(b):
    """Mean absolute miss, in signups a day, for each forecast week."""
    return (b["actual"] - b["forecast"]).abs().groupby(b["week"]).mean()


print(len(WEEKS), "full weeks,", len(WEEKS) - FIRST, "of them forecast")

## Unit 1: Trend, weekday pattern and noise

**Predict first.** Every weekday has a factor against the average day, and the factors average 1. Which day is the quietest, and is its factor about 0.6, about 0.8 or about 0.95?

In [ ]:
trend = signups.rolling(7, center=True).mean()                  # a window of one whole week: every weekday counts once, so the weekday pattern cancels
ratio = signups / trend                                             # what is left is the weekday pattern and the noise
factor = ratio.groupby(signups.index.dayofweek).mean()
factor = factor / factor.mean()                                     # weekday factors with an average of 1
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
readme = pd.Series([1.00, 1.05, 1.05, 1.00, 0.90, 0.75, 0.80], index=range(7))      # the weekday factors of the dataset README
readme = readme / readme.mean()
print(pd.DataFrame({"estimated": factor.round(3).to_numpy(), "README": readme.round(3).to_numpy()}, index=days).T.to_string())
noise = signups - trend * factor.reindex(signups.index.dayofweek).to_numpy()
print(round(float(noise.std()), 1), round(float(np.sqrt(signups.mean())), 1))      # the spread of what is left, and the square root of the average day

## Unit 2: A baseline, and a backtest that rolls

**Predict first.** The series has a strong weekday pattern. Will the seasonal naive forecast beat the flat last-7-days mean by a lot, by a little or not at all?

In [ ]:
def seasonal_naive(past, days):
    return past["y"].reindex(days - pd.Timedelta(days=7)).to_numpy()            # each day gets the value of the same weekday one week earlier


def last_week_mean(past, days):
    return np.full(len(days), past["y"].iloc[-7:].mean())                       # no weekday pattern: the average of the last 7 days for every day


b_naive, b_flat = backtest(seasonal_naive), backtest(last_week_mean)
print(len(weekly_mae(b_naive)), round(float(weekly_mae(b_naive).mean()), 2), round(float(weekly_mae(b_flat).mean()), 2))
print(round(float(df["y"].loc["2026-03-02":].mean()), 1))                        # the average day of the forecast weeks, for scale

## Unit 3: Features that exist on Sunday night

**Predict first.** The careless model adds `roll_now`, the mean of the 7 days that end on the day itself. Will its backtest error be above, about the same as or well below the honest model's?

In [ ]:
def week_features(y):
    """Features that exist on the Sunday night before a day's week: the same weekday last week and the mean of the last 7 days, plus the calendar."""
    origin = y.index - pd.to_timedelta(y.index.dayofweek + 1, unit="D")        # the Sunday before each day's week
    out = pd.DataFrame(index=y.index)
    out["lag7"] = y.shift(7)
    out["mean7"] = y.rolling(7).mean().reindex(origin).to_numpy()
    out["weekday"] = y.index.dayofweek
    out["month_start"] = (y.index.day <= 3).astype(float)                      # the 1st to the 3rd of a month: known in advance
    return out


def linear_forecaster(cols):
    def forecast(past, days):
        known = past["y"].reindex(past.index.union(days))                       # the days to forecast are empty here: nothing of theirs can leak in
        F = week_features(known)
        F["roll_now"] = df["y"].rolling(7).mean()                               # the careless feature: it reads the whole series, today included
        X = pd.get_dummies(F["weekday"].astype(int).astype(str), prefix="wd", dtype=float).reindex(columns=[f"wd_{i}" for i in range(7)], fill_value=0.0)
        X[["month_start"] + cols] = F[["month_start"] + cols]
        rows = X.index.isin(past.index) & ~X.isna().any(axis=1).to_numpy()
        model = LinearRegression().fit(X[rows], past["y"].reindex(X.index[rows]))
        return model.predict(X.loc[days])
    return forecast


b_model = backtest(linear_forecaster(["lag7", "mean7"]))
b_leaky = backtest(linear_forecaster(["lag7", "mean7", "roll_now"]))
print(round(float(weekly_mae(b_model).mean()), 2), round(float(weekly_mae(b_leaky).mean()), 2))

## Unit 4: Where the error comes from

**Predict first.** The paid-social campaign starts in the week of April 13. Will the model's error in that week be about its average, about twice it or under 10?

In [ ]:
README_DAY = [1.00, 1.05, 1.05, 1.00, 0.90, 0.75, 0.80]


def expected(past, days):
    """The dataset's own expected signups for each day: a straight line from 90 to 125, the weekday factor, 1.3 on days 1 to 3, and the paid-social campaign."""
    out = []
    for d in days:
        level = 90 + 35 * (d - df.index[0]).days / 180
        level *= README_DAY[d.weekday()] * (1.3 if d.day <= 3 else 1)
        if pd.Timestamp("2026-04-13") <= d <= pd.Timestamp("2026-06-07"):
            level += 55 * min(1.0, 0.4 + 0.6 * (d - pd.Timestamp("2026-04-13")).days / 7) * README_DAY[d.weekday()]
        out.append(level)
    return out


b_truth = backtest(expected)
weeks = pd.DataFrame({"seasonal naive": weekly_mae(b_naive), "model": weekly_mae(b_model), "expected": weekly_mae(b_truth)})
print(weeks.mean().round(2).to_string())
worst = weeks["model"].sort_values(ascending=False).head(3)
print(worst.round(1).to_string())
print(round(float(worst.sum() / weeks["model"].sum()), 3))
rng = np.random.default_rng(SEED)
diff = (weeks["seasonal naive"] - weeks["model"]).to_numpy()
means = [diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(2000)]      # resample whole weeks, not days
print(round(float(diff.mean()), 2), np.round(np.percentile(means, [2.5, 97.5]), 2))

## Finish it

### Exercise 1: Finish it

**Task:** **Two weeks instead of one.** The starter forecasts each day with the same weekday one week earlier. Add the same weekday two weeks earlier and average the two, then backtest it with `backtest` and `weekly_mae`. Put the mean weekly error over the 17 forecast weeks in `answer_1`, rounded to 2 decimals.

**Expected:** `answer_1` is a number.

In [ ]:
def two_week_naive(past, days):
    one_week_ago = past["y"].reindex(days - pd.Timedelta(days=7)).to_numpy()      # given: the same weekday one week earlier
    # add the same weekday two weeks earlier and average the two
    return one_week_ago


answer_1 = None  # replace None with the mean weekly MAE of this forecast over the 17 forecast weeks, rounded to 2 decimals

In [ ]:
_ = check("Exercise 1", "answer_1", 20.25, traps={20.92: "That is the seasonal naive forecast of one week. Average the weekday of one and two weeks earlier."}, tol=5e-3)

<details>
<summary>Hint</summary>

Use `past["y"].reindex(days - pd.Timedelta(days=14))` for the weekday two weeks earlier, and return the average of the two arrays.

</details>

<details>
<summary>Half-solution</summary>

```python
two_weeks_ago = past["y"].reindex(days - pd.Timedelta(days=...)).to_numpy()
return (one_week_ago + two_weeks_ago) / 2
```

</details>

<details>
<summary>Solution</summary>

```python
def two_week_naive(past, days):
    one_week_ago = past["y"].reindex(days - pd.Timedelta(days=7)).to_numpy()
    two_weeks_ago = past["y"].reindex(days - pd.Timedelta(days=14)).to_numpy()
    return (one_week_ago + two_weeks_ago) / 2


answer_1 = round(float(weekly_mae(backtest(two_week_naive)).mean()), 2)
```

Averaging two weeks gives 20.25 against 20.92 for one week alone: a little less noise is copied from the past, and the weeks around the campaign still dominate. A baseline can be tuned, and then it is the baseline to beat.

</details>

## On your own

### Exercise 2: On your own

**Task:** **Weeks with a month start.** Before you start, say in one sentence what you expect: a week that holds the 1st to the 3rd has a bump the seasonal naive forecast copies from the wrong place. Use `weeks` of unit 4: take the forecast weeks that contain a day from the 1st to the 3rd of a month, and put the list [mean weekly error of the seasonal naive forecast, mean weekly error of the model] over those weeks in `answer_2`, each rounded to 2 decimals. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of two numbers.

In [ ]:
# write your code here

In [ ]:
_ = check("Exercise 2", "answer_2", [22.86, 12.26], tol=5e-3)

<details>
<summary>Hint</summary>

`b_naive["day"]` holds every forecast day. Find the weeks with a day whose `.dt.day` is at most 3, then average those rows of `weeks`.

</details>

<details>
<summary>Half-solution</summary>

```python
month_start_week = b_naive.groupby("week")["day"].apply(lambda d: bool((d.dt.day <= 3).any()))
w = weeks.loc[...]
answer_2 = [round(float(w["seasonal naive"].mean()), 2), round(float(w["model"].mean()), 2)]
```

</details>

<details>
<summary>Solution</summary>

```python
month_start_week = b_naive.groupby("week")["day"].apply(lambda d: bool((d.dt.day <= 3).any()))
w = weeks.loc[month_start_week[month_start_week].index]
answer_2 = [round(float(w["seasonal naive"].mean()), 2), round(float(w["model"].mean()), 2)]
```

In the 4 weeks that hold a day from the 1st to the 3rd (the weeks of March 2, March 30, April 27 and June 1) the seasonal naive forecast misses by 22.86 a day and the model by 12.26. The model has the calendar flag, which is known in advance; the seasonal naive forecast copies a week that has no such bump, or copies the bump into a week that has none.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** **Spot the flaw.** A colleague fits one model on every day with a known answer, then scores it week by week from March 2, and reports the error of the starter. Put in `answer_3` the mean weekly error of the honest version: a new model before each of the 17 weeks, fitted on the days before that week only, with the same two features (`month_start` and `mean7`) and the weekday.

**Expected:** `answer_3` is a number.

In [ ]:
all_days = week_features(df["y"]).assign(y=df["y"]).dropna()
X_all = pd.get_dummies(all_days["weekday"].astype(int).astype(str), prefix="wd", dtype=float).reindex(columns=[f"wd_{i}" for i in range(7)], fill_value=0.0)
X_all[["month_start", "mean7"]] = all_days[["month_start", "mean7"]]
once = LinearRegression().fit(X_all, all_days["y"])             # colleague: "one model, fitted once on every day with a known answer, then scored week by week"
test_days = all_days.loc["2026-03-02":]
answer_3 = round(float((test_days["y"] - once.predict(X_all.loc[test_days.index])).abs().mean()), 2)
# fit a new model before each week, from the days before it only, with the same two features, and score the same 17 weeks

In [ ]:
_ = check("Exercise 3", "answer_3", 18.38, traps={16.1: "That is the model fitted once on every day. Fit a new model before each week."}, tol=5e-3)

<details>
<summary>Hint</summary>

`linear_forecaster` of unit 3 is the honest pattern: build the features for the past and the coming days together, fit on the past rows, predict the coming days. Use only `month_start` and `mean7`, and give the function to `backtest`.

</details>

<details>
<summary>Half-solution</summary>

```python
def mean7_forecaster(past, days):
    ...
    return LinearRegression().fit(X[rows], past["y"].reindex(X.index[rows])).predict(X.loc[days])


answer_3 = round(float(weekly_mae(backtest(mean7_forecaster)).mean()), 2)
```

</details>

<details>
<summary>Solution</summary>

```python
def mean7_forecaster(past, days):
    known = past["y"].reindex(past.index.union(days))
    F = week_features(known)
    X = pd.get_dummies(F["weekday"].astype(int).astype(str), prefix="wd", dtype=float).reindex(columns=[f"wd_{i}" for i in range(7)], fill_value=0.0)
    X[["month_start", "mean7"]] = F[["month_start", "mean7"]]
    rows = X.index.isin(past.index) & ~X.isna().any(axis=1).to_numpy()
    return LinearRegression().fit(X[rows], past["y"].reindex(X.index[rows])).predict(X.loc[days])


answer_3 = round(float(weekly_mae(backtest(mean7_forecaster)).mean()), 2)
```

The model fitted once on every day scores 16.1, and the honest version 18.38. The flawed model's coefficients were fitted on the weeks it is scored on, and a real forecast never has them. Fit before each origin, from the days before it.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")